# Notebook 02: Data Cleaning and Exploratory Data Analysis

This notebook validates the cleaned SentinelFlow dataset and generates basic exploratory summaries.

## Short Problem Statement

Existing deep learning-based **Intrusion Detection Systems (IDS)** often rely mainly on time-domain flow features and may miss hidden burst patterns, repeated traffic rhythms, and frequency-based attack signatures. **SentinelFlow** addresses this by using **Fast Fourier Transform (FFT)**-enhanced traffic profiling to improve deep learning-based intrusion detection on large-scale network traffic datasets.

In [ ]:
from pathlib import Path
import sys, json
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'sentinelflow_utils.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from sentinelflow_utils import *
ensure_dirs(PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
DATA_PATH = str(PROJECT_ROOT / 'data/processed/sentinelflow_analysis_dataset.parquet')
if not Path(DATA_PATH).exists():
    DATA_PATH = ''
path = find_dataset_path(PROJECT_ROOT, DATA_PATH)
print('Dataset path:', path)
df = read_table_safely(path)
print('Shape:', df.shape)
df.head()

In [ ]:
df, summary = clean_netflow_df(df)
print(summary)
cleaned_path = PROJECT_ROOT / 'data/processed/sentinelflow_cleaned_netflow.parquet'
df.to_parquet(cleaned_path, index=False)
print('Saved:', cleaned_path)

In [ ]:
binary_dist = class_distribution(df, 'target_binary')
attack_dist = class_distribution(df, 'target_attack')
display(binary_dist)
display(attack_dist.head(20))

In [ ]:
# Numeric feature summary for quick inspection.
numeric_cols = df.select_dtypes(include='number').columns.tolist()
summary_df = df[numeric_cols].describe().T.reset_index().rename(columns={'index':'feature'})
summary_df.to_csv(PROJECT_ROOT / 'outputs/eda_numeric_summary.csv', index=False)
summary_df.head(20)

In [ ]:
# Save an EDA summary HTML.
html = f"""
<html><head><title>SentinelFlow EDA Summary</title></head><body>
<h1>SentinelFlow EDA Summary</h1>
<p><b>Dataset shape:</b> {df.shape}</p>
<h2>Binary Class Distribution</h2>{binary_dist.to_html(index=False)}
<h2>Attack Class Distribution</h2>{attack_dist.head(30).to_html(index=False)}
<h2>Numeric Summary Sample</h2>{summary_df.head(30).to_html(index=False)}
</body></html>
"""
path = PROJECT_ROOT / 'reports/eda_summary.html'
path.write_text(html, encoding='utf-8')
print('Saved:', path)